# NVIDIA Armenia Jobs Scraping

**ATS:** Eightfold AI (`jobs.nvidia.com`)  
**Method:** Playwright loads the page to obtain a valid session, then calls the internal
`/api/pcsx/search` endpoint (discovered via network interception) with `location=Yerevan, Armenia`.
Detail text fetched via `/api/pcsx/position_details`.  
**Why Playwright:** Eightfold requires browser-generated session cookies; plain HTTP returns 403.  
**Outputs:** `data/raw/jobs/nvidia_jobs_raw.csv`, `data/processed/jobs/nvidia_jobs_standardized.csv`

In [1]:
from pathlib import Path, Path as P
import os

_nb = globals().get('__vsc_ipynb_file__') or globals().get('__file__', None)
if _nb:
    os.chdir(P(_nb).resolve().parent.parent.parent)
elif not (P.cwd() / 'data').exists():
    _r = P.cwd()
    for _ in range(4):
        if (_r / 'data').exists(): break
        _r = _r.parent
    os.chdir(_r)
assert (P.cwd() / 'data').exists(), f'Cannot find root from {P.cwd()}'
RAW_DIR  = P.cwd() / 'data/raw/jobs'
PROC_DIR = P.cwd() / 'data/processed/jobs'
print('Root:', P.cwd())

Root: /Users/lianaaghamalyan/thesis_data


In [2]:
import re, asyncio, json, pandas as pd
from bs4 import BeautifulSoup
from playwright.async_api import async_playwright

BASE = 'https://jobs.nvidia.com'

def html_to_text(h):
    if not h: return ''
    t = BeautifulSoup(str(h), 'html.parser').get_text('\n', strip=True)
    return re.sub(r'\n{3,}', '\n\n', t).strip()

print(f'Base: {BASE}')

Base: https://jobs.nvidia.com


In [3]:
async def scrape_nvidia():
    records = []
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()

        # Load careers page to establish a valid session
        print('Loading NVIDIA careers page to establish session...')
        await page.goto(f'{BASE}/careers', wait_until='networkidle', timeout=40000)
        await page.wait_for_timeout(2000)

        # Step 1: Search for Yerevan jobs via internal PCSX API
        print('Querying PCSX search API for Yerevan, Armenia...')
        search_result = await page.evaluate("""
            async () => {
                const resp = await fetch('/api/pcsx/search?domain=nvidia.com&query=&location=Yerevan%2C+Armenia&start=0&num=100', {
                    credentials: 'include',
                    headers: {'Accept': 'application/json'}
                });
                return await resp.json();
            }
        """)

        positions = search_result.get('data', {}).get('positions', [])
        print(f'Armenia/Yerevan jobs found: {len(positions)}')
        for pos in positions:
            locs = ', '.join(pos.get('locations', []))
            print(f"  {pos['name']:<55} | {locs}")

        # Step 2: Fetch detail for each job
        for i, pos in enumerate(positions, 1):
            pos_id = pos['id']
            title = pos.get('name', '')
            print(f'\n[{i}/{len(positions)}] {title}')

            try:
                detail = await page.evaluate(f"""
                    async () => {{
                        const resp = await fetch('/api/pcsx/position_details?position_id={pos_id}&domain=nvidia.com&hl=en', {{
                            credentials: 'include',
                            headers: {{'Accept': 'application/json'}}
                        }});
                        return await resp.json();
                    }}
                """)

                detail_data = detail.get('data', {})
                description_html = (detail_data.get('description') or
                                    detail_data.get('jobDescription') or
                                    detail_data.get('text') or '')
                full_text = html_to_text(description_html) if description_html else ''

                # Fallback: concatenate all string fields
                if not full_text:
                    parts = [str(v) for v in detail_data.values()
                             if isinstance(v, str) and len(v) > 50]
                    full_text = '\n\n'.join(parts)

                print(f'  {len(full_text)} chars | keys: {list(detail_data.keys())[:6]}')

                loc_str = ', '.join(pos.get('locations', []))
                records.append({
                    'source': 'nvidia',
                    'source_type': 'company_portal',
                    'source_url': f"{BASE}{pos.get('positionUrl', '')}",
                    'job_title': title,
                    'company_name': 'NVIDIA',
                    'location': loc_str,
                    'department': pos.get('department', ''),
                    'employment_type': '',
                    'seniority_level': '',
                    'industries': 'Technology / Semiconductors',
                    'posting_date': '',
                    'skills_tags': '',
                    'full_text': full_text,
                })
            except Exception as e:
                print(f'  Error: {e}')
            await page.wait_for_timeout(1000)

        await browser.close()
    return records

records = await scrape_nvidia()

Loading NVIDIA careers page to establish session...
Querying PCSX search API for Yerevan, Armenia...
Armenia/Yerevan jobs found: 5
  Applied Research Intern - 2026                          | Armenia, Yerevan
  Senior Data Engineer                                    | Poland, Warsaw, Hungary, Budapest, Armenia, Yerevan
  Senior Full Stack Engineer                              | Poland, Warsaw, Hungary, Budapest, Armenia, Yerevan, Remote - Poland
  Senior Software Engineer, Real-Time Data Platform       | Remote - Poland, Poland, Warsaw, Remote - Armenia, Armenia, Yerevan
  Research Scientist - New College Grad 2025              | Armenia, Yerevan

[1/5] Applied Research Intern - 2026
  1294 chars | keys: ['id', 'displayJobId', 'name', 'locations', 'standardizedLocations', 'postedTs']

[2/5] Senior Data Engineer
  2891 chars | keys: ['id', 'displayJobId', 'name', 'locations', 'standardizedLocations', 'postedTs']

[3/5] Senior Full Stack Engineer
  3327 chars | keys: ['id', 'displayJobId'

In [4]:
df = pd.DataFrame(records)
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROC_DIR.mkdir(parents=True, exist_ok=True)
df.to_csv(RAW_DIR  / 'nvidia_jobs_raw.csv', index=False)
df.to_csv(PROC_DIR / 'nvidia_jobs_standardized.csv', index=False)
print(f'Saved {len(df)} rows')
if len(df):
    print(df[['job_title', 'location', 'department']].to_string())

Saved 5 rows
                                           job_title                                                              location             department
0                     Applied Research Intern - 2026                                                      Armenia, Yerevan                 Intern
1                               Senior Data Engineer                   Poland, Warsaw, Hungary, Budapest, Armenia, Yerevan       Engineer, Sys SW
2                         Senior Full Stack Engineer  Poland, Warsaw, Hungary, Budapest, Armenia, Yerevan, Remote - Poland       Engineer, Sys SW
3  Senior Software Engineer, Real-Time Data Platform   Remote - Poland, Poland, Warsaw, Remote - Armenia, Armenia, Yerevan       Engineer, Sys SW
4         Research Scientist - New College Grad 2025                                                      Armenia, Yerevan  Research Staff Member
